# A Jagged Cut

### Reading Visual Archives at Multiple Resolutions with HDBSCAN

Companion notebook to the paper. Cells run top to bottom and follow the order of
the argument: embed, reduce, cluster, recover the hierarchy, compute the three
measures, then draw the figures.

Data is not in this repository — see the README for where to get it.

## Setup

Imports, paths, and the pipeline settings that stay fixed throughout. `SUFFIX`
is appended to every file this notebook writes.

In [ ]:
import json, itertools, time, base64, random
from pathlib import Path
from io import BytesIO
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch
from PIL import Image
from scipy.stats import spearmanr

from umap import UMAP
from hdbscan import HDBSCAN
from hdbscan.validity import validity_index

np.random.seed(42)
random.seed(42)

# ---- paths -------------------------------------------------------------
# Two inputs, neither in this repository. The embeddings come from
# https://doi.org/10.5281/zenodo.20847435; the images and metadata are
# downloaded by notebook 01. See the README.
DATA_DIR      = Path("../data")
IMAGE_FOLDER  = DATA_DIR / "images"
CLIP_FEATURES = DATA_DIR / "massafont_clip_features_vitl14P_l2b.npy"
CLIP_PATHS    = DATA_DIR / "massafont_clip_paths.json"   # cache, written on first run
METADATA_CSV  = DATA_DIR / "massafont_metadata.csv"      # from notebook 01
FIG_DIR       = Path("../figures")
FIG_DIR.mkdir(exist_ok=True)

# ---- fixed pipeline settings ------------------------------------------
UMAP_METRIC       = "cosine"
UMAP_MIN_DIST     = 0.0
UMAP_RANDOM_STATE = 42
MIN_CLUSTER_SIZE  = 20

SUFFIX = "clip"
print("imports OK")

## 1. Load embeddings and align image paths

The embeddings were extracted separately with OpenCLIP ViT-L/14 (LAION-2B). The
path list must be in the same order as the feature matrix; the assert catches the
case where the image folder changed after extraction, which is not recoverable.

In [ ]:
extensions = ("*.jpg", "*.jpeg", "*.JPG", "*.JPEG")

if Path(CLIP_PATHS).exists():
    paths = json.loads(Path(CLIP_PATHS).read_text())
    print(f"loaded cached path list: {len(paths)}")
else:
    glob_paths = []
    for ext in extensions:
        glob_paths += list(Path(IMAGE_FOLDER).rglob(ext))
    paths = sorted(str(p) for p in glob_paths)
    Path(CLIP_PATHS).write_text(json.dumps(paths))
    print(f"globbed {len(paths)} paths -> {CLIP_PATHS}")

if not Path(CLIP_FEATURES).exists():
    raise FileNotFoundError(
        f"{CLIP_FEATURES} not found.\n"
        "Download the embeddings from https://doi.org/10.5281/zenodo.20847435 "
        f"and place them in {DATA_DIR.resolve()}, or edit CLIP_FEATURES above.")

embeddings = np.load(CLIP_FEATURES)
assert len(paths) == len(embeddings), (
    f"path list ({len(paths)}) does not match embeddings ({len(embeddings)}); "
    "the image folder has changed since the features were computed")
print(f"embeddings: {embeddings.shape}")

## 2. Pipeline helpers

`apply_umap` and `dbcv` are used by both the sweep and the final run, so the two
are guaranteed to be doing the same thing. The assert verifies that the features
are unit length rather than assuming it.

In [ ]:
def apply_umap(X, n_components, n_neighbors, random_state=UMAP_RANDOM_STATE):
    return UMAP(n_components=n_components, n_neighbors=n_neighbors,
                metric=UMAP_METRIC, min_dist=UMAP_MIN_DIST,
                random_state=random_state).fit_transform(X).astype(np.float32)


def dbcv(red, lab):
    """Density-Based Clustering Validation, computed on clustered points only."""
    if len(set(lab) - {-1}) < 2 or np.mean(lab == -1) > 0.95:
        return np.nan
    m = lab != -1
    try:
        return float(validity_index(red[m].astype(np.float64), lab[m]))
    except Exception:
        return np.nan


def center_crop_square(img):
    """Square centre crop, used for every thumbnail in the figures below."""
    w, h = img.size
    s = min(w, h)
    return img.crop(((w - s) // 2, (h - s) // 2, (w + s) // 2, (h + s) // 2))


norms = np.linalg.norm(embeddings, axis=1)
assert np.allclose(norms, 1.0, atol=1e-3), \
    "embeddings are not unit length - add an L2 normalisation step before UMAP"

X = embeddings.astype(np.float32)
print(f"ready: {X.shape}")

## 3. Parameter sweep

27 configurations scored with DBCV. Set `RUN_SWEEP = False` to skip; the chosen
configuration is hard-coded in the next cell either way, so the sweep only needs
running once.

In [ ]:
RUN_SWEEP = False

GRID = list(itertools.product(
    [10, 15, 25],   # n_components
    [15, 30, 50],   # n_neighbors
    [5, 10, 20],    # min_samples
))

if RUN_SWEEP:
    rows = []
    print(f"sweeping {len(GRID)} configurations")
    for i, (nc, nn, ms) in enumerate(GRID, 1):
        t0 = time.time()
        red = apply_umap(X, nc, nn)
        lab = HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE, min_samples=ms,
                      metric="euclidean").fit_predict(red)
        rows.append(dict(n_components=nc, n_neighbors=nn, min_samples=ms,
                         n_clusters=len(set(lab) - {-1}),
                         noise=round(float(np.mean(lab == -1)), 3),
                         dbcv=round(dbcv(red, lab), 4)))
        print(f"  [{i}/{len(GRID)}] nc={nc} nn={nn} ms={ms} -> "
              f"k={rows[-1]['n_clusters']}, noise={rows[-1]['noise']:.0%}, "
              f"DBCV={rows[-1]['dbcv']}  ({time.time()-t0:.0f}s)")

    sweep = pd.DataFrame(rows).sort_values("dbcv", ascending=False).reset_index(drop=True)
    sweep.to_csv(f"sweep_{SUFFIX}.csv", index=False)
    print("\n=== ranked by DBCV ===")
    print(sweep.to_string(index=False))
    print("\n=== mean DBCV by min_samples ===")
    print(sweep.groupby("min_samples")["dbcv"].mean().round(4).to_string())
else:
    print("sweep skipped; using the configuration below")

## 4. Final clustering

The best-scoring configuration: 25 dimensions, neighbourhood 30,
`min_samples = min_cluster_size = 20`. Produces 72 clusters and 27% noise.

In [ ]:
BEST = dict(n_components=25, n_neighbors=30, min_samples=20)

reduced = apply_umap(X, BEST["n_components"], BEST["n_neighbors"])

clusterer = HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE,
                    min_samples=BEST["min_samples"],
                    metric="euclidean", gen_min_span_tree=True,
                    prediction_data=True)
labels = clusterer.fit_predict(reduced)

unique_labels = sorted(set(labels) - {-1})
n_samples = len(labels)
n_noise = int(np.sum(labels == -1))
print(f"{len(unique_labels)} clusters, {n_noise} noise ({n_noise/n_samples:.1%})")
print(f"DBCV: {dbcv(reduced, labels):.4f}")

np.save(f"reduced_{SUFFIX}.npy", reduced)
np.save(f"labels_{SUFFIX}.npy", labels)
print(f"saved -> reduced_{SUFFIX}.npy, labels_{SUFFIX}.npy")

## 5. Recover the hierarchy from the condensed tree

Three steps. Match each of the 72 clusters to the node holding exactly its
photographs and no others. Keep the merge events above those nodes (the
*backbone*). Then descend below them through single-child shedding chains to the
next genuine two-way split (the *sub-structure*).

`SUB_DEPTH = None` descends until no further splits exist.

In [ ]:
condensed_tree_df = clusterer.condensed_tree_.to_pandas()

raw_children = defaultdict(list)
for _, r in condensed_tree_df.iterrows():
    raw_children[int(r.parent)].append(int(r.child))

node_lambda = {int(r.child): float(r.lambda_val)
               for _, r in condensed_tree_df.iterrows() if int(r.child) >= n_samples}

memo_size = {}
def node_point_count(nid):
    if nid < n_samples:
        return 1
    if nid in memo_size:
        return memo_size[nid]
    memo_size[nid] = sum(node_point_count(c) for c in raw_children.get(nid, []))
    return memo_size[nid]


def node_point_set(nid):
    out, stack = set(), [nid]
    while stack:
        x = stack.pop()
        if x < n_samples:
            out.add(x)
        else:
            stack.extend(raw_children.get(x, []))
    return out


for nid in sorted({int(p) for p in condensed_tree_df.parent}):
    node_point_count(nid)

# ---- match each cluster to its node ------------------------------------
cluster_node_id, birth_height = {}, {}
for L in unique_labels:
    pts = set(np.where(labels == L)[0])
    cands = [nid for nid, sz in memo_size.items() if sz == len(pts)]
    match = next((nid for nid in cands if node_point_set(nid) == pts), None)
    if match is None:
        print(f"WARNING: no exact node match for cluster {L}")
    cluster_node_id[L] = match
    m = condensed_tree_df[condensed_tree_df["child"] == match]
    birth_height[L] = float(m["lambda_val"].values[0]) if len(m) else 0.0
print(f"resolved {sum(v is not None for v in cluster_node_id.values())} "
      f"of {len(unique_labels)} clusters")

# ---- backbone: everything above the selected nodes ---------------------
def descendants(nid):
    stack, seen = [nid], set()
    while stack:
        x = stack.pop()
        seen.add(x)
        stack.extend(condensed_tree_df[condensed_tree_df["parent"] == x]["child"].tolist())
    return seen


below = set()
for nid in cluster_node_id.values():
    if nid is not None:
        below |= descendants(nid) - {nid}

backbone = condensed_tree_df[
    (~condensed_tree_df["child"].isin(below)) &
    (condensed_tree_df["child"] >= n_samples)].copy()

children_of = {}
for _, r in backbone.iterrows():
    children_of.setdefault(int(r["parent"]), []).append(int(r["child"]))
node_height = {int(r["child"]): float(r["lambda_val"]) for _, r in backbone.iterrows()}
root = int(backbone.loc[~backbone["parent"].isin(backbone["child"]), "parent"].iloc[0])
node_height[root] = 0.0
leaf_of_node = {nid: L for L, nid in cluster_node_id.items() if nid is not None}
child_to_parent = {c: p for p, kids in children_of.items() for c in kids}
print(f"backbone: {len(backbone)} merge events")

# ---- sub-structure: everything below the selected nodes ----------------
SUB_DEPTH = None                    # None = descend all the way
MIN_SUB_SIZE = MIN_CLUSTER_SIZE     # redundant by construction, kept explicit

def next_split(nid, guard=10_000):
    """Descend through single-child shedding chains to the next two-way split."""
    cur = nid
    for _ in range(guard):
        kids = [c for c in raw_children.get(cur, [])
                if c >= n_samples and node_point_count(c) >= MIN_SUB_SIZE]
        if len(kids) == 0:
            return None
        if len(kids) == 1:
            cur = kids[0]
            continue
        return kids
    return None


sub_children_of, sub_height, sub_size, sub_owner, sub_depth_of = {}, {}, {}, {}, {}
for L, nid in cluster_node_id.items():
    if nid is None:
        continue
    frontier = [(nid, 0)]
    while frontier:
        node, d = frontier.pop()
        if SUB_DEPTH is not None and d >= SUB_DEPTH:
            continue
        kids = next_split(node)
        if kids is None:
            continue
        sub_children_of[node] = kids
        for c in kids:
            sub_height[c], sub_size[c] = node_lambda[c], node_point_count(c)
            sub_owner[c], sub_depth_of[c] = L, d + 1
            frontier.append((c, d + 1))

have_sub = [L for L, nid in cluster_node_id.items() if nid in sub_children_of]
depths = defaultdict(int)
for c, d in sub_depth_of.items():
    depths[d] += 1
print(f"sub-structure: {len(have_sub)} of {len(cluster_node_id)} clusters split further")
print(f"  sub-nodes per depth: {dict(sorted(depths.items()))}")
if sub_height:
    print(f"  lambda range of sub-nodes: {min(sub_height.values()):.2f} - "
          f"{max(sub_height.values()):.2f}   "
          f"(max cluster birth: {max(birth_height.values()):.2f})")
print("  clusters with sub-structure:", ", ".join(str(L) for L in sorted(have_sub)))

## 6. Order the clusters and lay out the tree

A dendrogram's horizontal axis carries no information: sibling branches can be
drawn in any order. We use that freedom to place similar clusters next to one
another, minimising embedding distance at the seams between sibling groups.
Exhaustive below `MAX_EXACT` children, greedy above it.

Sub-clusters are laid out in a narrow band beneath their parent, so adjacency
below the cut should *not* be read the same way.

In [ ]:
MAX_EXACT = 5
SUB_BAND  = 0.75

labels_sorted = sorted(cluster_node_id)
lab_ix = {L: i for i, L in enumerate(labels_sorted)}
C = np.vstack([embeddings[labels == L].mean(axis=0) for L in labels_sorted])
C = C / np.linalg.norm(C, axis=1, keepdims=True)
D = 1.0 - C @ C.T


def _first(s): return next((v for k, v in s if k == "leaf"), None)
def _last(s):  return next((v for k, v in reversed(s) if k == "leaf"), None)


def ordered_leaves(node):
    if node in leaf_of_node:
        return [("leaf", leaf_of_node[node])]
    if node not in children_of:
        return [("dangle", node)]
    kids = [ordered_leaves(k) for k in children_of[node]]
    if len(kids) == 1:
        return kids[0]

    def cost(s):
        t = 0.0
        for j in range(len(s) - 1):
            a, b = _last(s[j]), _first(s[j + 1])
            if a is not None and b is not None:
                t += D[lab_ix[a], lab_ix[b]]
        return t

    if len(kids) <= MAX_EXACT:
        best, bc = None, np.inf
        for p in itertools.permutations(range(len(kids))):
            for fl in itertools.product([False, True], repeat=len(kids)):
                cand = [kids[p[j]][::-1] if fl[j] else kids[p[j]] for j in range(len(kids))]
                c = cost(cand)
                if c < bc:
                    best, bc = cand, c
    else:
        real, empty, cents = [], [], []
        for i, kl in enumerate(kids):
            lb = [v for k, v in kl if k == "leaf"]
            if lb:
                real.append(i)
                cents.append(np.mean([C[lab_ix[L]] for L in lb], axis=0))
            else:
                empty.append(i)
        cents = np.vstack(cents)
        cents = cents / np.linalg.norm(cents, axis=1, keepdims=True)
        Dk = 1.0 - cents @ cents.T
        rem, order = set(range(len(real))), [0]
        rem.remove(0)
        while rem:
            nxt = min(rem, key=lambda j: Dk[order[-1], j])
            order.append(nxt); rem.remove(nxt)
        best = [kids[real[i]] for i in order] + [kids[i] for i in empty]
    return [x for sub in best for x in sub]


seq = ordered_leaves(root)
leaf_order = [v for k, v in seq if k == "leaf"]
x_pos = {}
for i, (k, v) in enumerate(seq):
    x_pos[cluster_node_id[v] if k == "leaf" else v] = i

_span = {}
def span(nid):
    if nid in _span:
        return _span[nid]
    if nid not in children_of:
        r = (x_pos[nid], x_pos[nid])
    else:
        ks = [span(k) for k in children_of[nid]]
        r = (min(k[0] for k in ks), max(k[1] for k in ks))
    _span[nid] = r
    return r


for nid in children_of:
    lo, hi = span(nid)
    x_pos[nid] = (lo + hi) / 2


def layout_sub(node, lo, hi):
    kids = sub_children_of.get(node)
    if not kids:
        return
    kids = sorted(kids, key=lambda k: -sub_size[k])
    tot = sum(sub_size[k] for k in kids)
    cur = lo
    for k in kids:
        w = (hi - lo) * sub_size[k] / tot
        x_pos[k] = cur + w / 2
        layout_sub(k, cur, cur + w)
        cur += w


for L, nid in cluster_node_id.items():
    if nid in sub_children_of:
        layout_sub(nid, x_pos[nid] - SUB_BAND / 2, x_pos[nid] + SUB_BAND / 2)

missing = set(sub_size) - set(x_pos)
assert not missing, f"{len(missing)} sub-nodes lack x_pos - rerun cell 5 then this one"
print(f"ordered {len(leaf_order)} clusters; laid out {len(sub_size)} sub-nodes")

## 7. Traversal helpers

Small functions used by every measure and figure below: which clusters sit under
a node, which photographs, and at what density a node comes into being.

In [ ]:
def backbone_leaves(nid):
    """Selected clusters (as node ids) beneath a backbone node."""
    if nid in leaf_of_node:
        return [nid]
    return [x for k in children_of.get(nid, []) for x in backbone_leaves(k)]


def branch_images(nid):
    """Indices of all clustered photographs beneath a backbone node."""
    ls = [leaf_of_node[l] for l in backbone_leaves(nid)]
    if not ls:
        return []
    return sorted(np.concatenate([np.where(labels == L)[0] for L in ls]))


def n_img(nid):
    return len(branch_images(nid))


def birth_of(nid):
    """Lambda at which a node comes into being, above or below the cut."""
    if nid in sub_height:
        return sub_height[nid]
    if nid in leaf_of_node:
        return birth_height[leaf_of_node[nid]]
    return node_height.get(nid, 0.0)


def subtree_nodes(nid):
    out, stack = set(), [nid]
    while stack:
        x = stack.pop(); out.add(x)
        stack.extend(children_of.get(x, []))
    return out


print("helpers defined")

## 8. Measure 1 — Specificity

The lambda at which a cluster is born: the density at which it becomes
distinguishable from what surrounds it. High means a scene type that separates
while the density is still tight.

The second half tests whether specificity is just a proxy for cluster size. It
is not (Spearman rho = -0.08, p = 0.51).

In [ ]:
tree_stats = pd.DataFrame([
    dict(cluster=L, size=int(np.sum(labels == L)),
         lambda_birth=round(birth_height[L], 3),
         has_substructure=cluster_node_id[L] in sub_children_of)
    for L in unique_labels])

spec = tree_stats.sort_values("lambda_birth", ascending=False)
cols = ["cluster", "size", "lambda_birth", "has_substructure"]
print("=== most specific ===")
print(spec.head(4)[cols].to_string(index=False))
print("\n=== least specific ===")
print(spec.tail(4)[cols].iloc[::-1].to_string(index=False))

rho, p = spearmanr(tree_stats["size"], tree_stats["lambda_birth"])
print(f"\nsize vs specificity: Spearman rho = {rho:+.3f}, p = {p:.3f}, "
      f"n = {len(tree_stats)}")

tree_stats.to_csv(f"specificity_{SUFFIX}.csv", index=False)

## 9. Measure 2 — Load-bearing splits

Every internal node marks a division; this scores each by the number of
photographs in its *smaller* branch, so a split ranks highly only when both
sides are substantial. `score` equals `smallest` by construction — `total x
balance` is the same quantity written differently.

In [ ]:
rows = []
for nid, kids in children_of.items():
    if len(kids) < 2:
        continue
    sizes = [n_img(k) for k in kids]
    tot = sum(sizes)
    if tot == 0:
        continue
    rows.append(dict(node=nid, lam=round(node_height[nid], 3),
                     total_size=tot, smallest=min(sizes), largest=max(sizes),
                     balance=round(min(sizes) / tot, 3), score=min(sizes)))

splits = pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)
splits.to_csv(f"load_bearing_splits_{SUFFIX}.csv", index=False)

print("=== most load-bearing splits ===")
print(splits.head(8).to_string(index=False))

print("\n=== what lies on each side of the top three ===")
for _, r in splits.head(3).iterrows():
    kids = sorted(children_of[int(r.node)], key=lambda k: -n_img(k))
    print(f"\nnode {int(r.node)} (lambda {r.lam})")
    for k in kids:
        ls = sorted(int(leaf_of_node[l]) for l in backbone_leaves(k))
        print(f"  {n_img(k):>5} photographs, {len(ls):>2} clusters: {ls}")

## 10. Measure 3 — Attachment

Every unclustered photograph left the tree at exactly one node. That node's
lambda, and the number of clusters still beneath it, say how close the photograph
came to belonging: shed above one or two clusters is a near miss, shed near the
root means no group of similar images formed around it at any density.

In [ ]:
# every photograph appears once as a child: the node it left, and when
point_fall = {}
for _, r in condensed_tree_df.iterrows():
    c = int(r.child)
    if c < n_samples:
        point_fall[c] = (int(r.parent), float(r.lambda_val))

rows = []
for i in np.where(labels == -1)[0]:
    par, lam = point_fall.get(int(i), (None, np.nan))
    rows.append({"idx": int(i), "node": par, "exit_lambda": lam})
noise_df = pd.DataFrame(rows)
noise_df["beneath"] = noise_df.node.map(
    lambda n: len(backbone_leaves(n)) if n in node_height else np.nan)
noise_df["node_lambda"] = noise_df.node.map(lambda n: node_height.get(n, np.nan))

n_noise = len(noise_df)
print(f"{n_noise} unclustered photographs ({n_noise/n_samples:.1%} of the collection)")
print(f"shed at {noise_df.node.nunique()} distinct junctions\n")

# ---- Table 3 in the paper ---------------------------------------------
# band edges built from the actual cluster count, so they stay monotonic
n_cl = len(unique_labels)
cuts = [c for c in [2, 5, 10, 20, 40] if c < n_cl] + [n_cl]
bins, names, lo = [0], [], 1
for c in cuts:
    bins.append(c)
    names.append(f"{lo}-{c}" if c > lo else str(lo))
    lo = c + 1
noise_df["band"] = pd.cut(noise_df.beneath, bins=bins, labels=names, include_lowest=True)

tab = (noise_df.groupby("band", observed=True)
       .agg(photographs=("idx", "size"), junctions=("node", "nunique"),
            median_lambda=("node_lambda", "median")))
tab["pct"] = (tab.photographs / n_noise * 100).round(1)
tab["cumulative"] = tab.pct.cumsum().round(1)
tab["median_lambda"] = tab.median_lambda.round(2)
print("attachment, by clusters beneath the shedding junction:")
print(tab[["photographs", "pct", "cumulative", "junctions", "median_lambda"]].to_string())

for k in [2, 10]:
    s = (noise_df.beneath <= k).mean()
    print(f"\nshed above {k} clusters or fewer: {int(s*n_noise)} ({s:.1%} of noise)")
top_band = noise_df.beneath.max()
n_root = int((noise_df.beneath == top_band).sum())
print(f"shed above {int(top_band)} clusters: {n_root} "
      f"({n_root/n_noise:.1%} of noise, {n_root/n_samples:.1%} of the collection)")

noise_df.to_csv(f"attachment_{SUFFIX}.csv", index=False)

## 11. Attachment scored per junction

The same quantity read the other way round: of the photographs arriving at a
junction, what proportion left there rather than passing on. A high ratio marks a
region the algorithm could barely organise; a low one a region that resolved
cleanly. Restricted to junctions above three clusters or fewer, where the ratio
describes a specific scene type rather than a whole branch.

In [ ]:
near = noise_df[noise_df.beneath <= 3]
rows = []
for nd, sub in near.groupby("node"):
    ls = sorted(int(leaf_of_node[l]) for l in backbone_leaves(nd))
    clustered = int(sum(np.sum(labels == L) for L in ls))
    rows.append({"node": int(nd), "lam": round(node_height[nd], 2),
                 "shed": len(sub), "clustered": clustered,
                 "ratio": round(len(sub) / (len(sub) + clustered), 3),
                 "clusters": ",".join(map(str, ls))})
near_df = pd.DataFrame(rows).sort_values("ratio", ascending=False)

print(f"{len(near_df)} junctions above <=3 clusters, "
      f"{int(near_df.shed.sum())} photographs "
      f"({near_df.shed.sum()/n_noise:.1%} of noise)\n")
print("=== highest shed ratio ===")
print(near_df.head(5).to_string(index=False))
print("\n=== lowest shed ratio ===")
print(near_df.tail(5).to_string(index=False))

near_df.to_csv(f"shed_ratio_{SUFFIX}.csv", index=False)

## 12. Inspection helpers

Used while interpreting, not for the final figures. `show_cluster_all` and
`show_shed_all` render every image in a cluster or at a shedding junction;
`show_shed_indexed` labels each with its position in `paths`, which is how the
cowboys photograph in Figure 1 was located.

In [ ]:
def _grid(idx, title, cols=10, size=1.6, labels_on=False):
    print(title)
    rows_n = int(np.ceil(len(idx) / cols))
    fig, axes = plt.subplots(rows_n, cols, figsize=(cols * size, rows_n * size * 1.1))
    for ax, i in zip(np.ravel(axes), idx):
        ax.imshow(center_crop_square(Image.open(paths[i])).convert("L"), cmap="gray")
        if labels_on:
            ax.set_title(str(i), fontsize=6, pad=1)
        ax.axis("off")
    for ax in np.ravel(axes)[len(idx):]:
        ax.axis("off")
    plt.tight_layout(); plt.show()


def show_cluster_all(L, **kw):
    idx = np.where(labels == L)[0]
    _grid(idx, f"cluster {L}  |  lambda={birth_height[L]:.2f}  |  {len(idx)} images", **kw)


def show_shed_all(node, **kw):
    idx = noise_df.loc[noise_df.node == node].sort_values(
        "exit_lambda", ascending=False).idx.tolist()
    ls = sorted(int(leaf_of_node[l]) for l in backbone_leaves(node))
    _grid(idx, f"node {node}  |  lambda={node_height[node]:.2f}  |  {len(idx)} shed  |  "
               f"clusters beneath: {ls}", **kw)


def show_shed_indexed(node, **kw):
    show_shed_all(node, labels_on=True, **kw)


def show_subcluster(nid, **kw):
    _grid(sorted(node_point_set(nid)),
          f"sub-cluster {nid} of cluster {sub_owner[nid]}  |  "
          f"lambda={sub_height[nid]:.2f}  |  {sub_size[nid]} images", **kw)


print("inspection helpers defined")

# Figures

Everything below produces output for the paper, in the order the figures appear.
All are written as PDF (vector, for print) and PNG (fallback).

Sizes are set for a `\textwidth` of about 6.75 in and a `\textheight` of about
9 in — measure `\the\textwidth` in the document and adjust `FIG_W` if the
template differs. Fonts are set in real points and the figure is drawn at final
size, so nothing is rescaled on the way into LaTeX.

In [ ]:
FIG_W       = 6.75      # \textwidth in inches
FIG_W_WIDE  = 9.0       # \textheight, for rotated full-page figures
BASE_PT     = 8
TICK_PT     = 3.4

PAPER_RC = {
    "font.family": "serif", "font.serif": ["DejaVu Serif"],
    "font.size": BASE_PT, "axes.labelsize": BASE_PT,
    "legend.fontsize": BASE_PT - 1,
    "xtick.labelsize": TICK_PT, "ytick.labelsize": BASE_PT - 1,
    "axes.linewidth": 0.5,
    "xtick.major.size": 1.2, "ytick.major.size": 2.0,
    "xtick.major.width": 0.35, "ytick.major.width": 0.4,
    "xtick.major.pad": 1.5, "pdf.fonttype": 42, "ps.fonttype": 42,
}


def save_fig(out, pad=0.02):
    out = FIG_DIR / out
    plt.savefig(f"{out}.pdf", facecolor="white", bbox_inches="tight", pad_inches=pad)
    plt.savefig(f"{out}.png", dpi=600, facecolor="white",
                bbox_inches="tight", pad_inches=pad)
    plt.show()
    print(f"saved -> {out}.pdf (and .png)")


print("figure defaults set")

## Band figures

Most picture figures in the paper are stacked bands of thumbnails with a heading
above each. `band_figure` takes `(indices, label)` pairs; `climb_figure` takes
`(indices, label, lambda)` and reads bottom to top, with arrows linking the
lambda values.

In [ ]:
def band_figure(bands, out, w_in=FIG_W, n_per_row=8, gap=0.03,
                head=0.16, band_gap=0.06, label_pt=BASE_PT, seed=0):
    """Stacked bands of thumbnails, each with a heading above it."""
    mpl.rcParams.update(PAPER_RC)
    thumb = (w_in - gap * (n_per_row - 1)) / n_per_row
    h_in = len(bands) * (thumb + head) + band_gap * (len(bands) - 1)
    print(f"thumbnail {thumb*25.4:.0f}mm   figure {w_in:.2f} x {h_in:.2f}in")

    rng = np.random.default_rng(seed)
    fig = plt.figure(figsize=(w_in, h_in), facecolor="white")
    y = h_in
    for idx, label in bands:
        idx = np.asarray(idx)
        pick = rng.choice(idx, size=min(n_per_row, len(idx)), replace=False)
        fig.text(0.0, (y - head * 0.72) / h_in, label,
                 ha="left", va="baseline", fontsize=label_pt)
        y -= head
        y_img = y - thumb
        for col, i in enumerate(pick):
            ax = fig.add_axes([col * (thumb + gap) / w_in, y_img / h_in,
                               thumb / w_in, thumb / h_in])
            ax.imshow(center_crop_square(Image.open(paths[i])).convert("L"), cmap="gray")
            ax.set_xticks([]); ax.set_yticks([])
            for s in ax.spines.values():
                s.set_linewidth(0.3); s.set_color("#cccccc")
        y = y_img - band_gap

    save_fig(out, pad=0.01)
    mpl.rcParams.update(mpl.rcParamsDefault)


def climb_figure(bands, out, w_in=FIG_W, n_per_row=8, gap=0.03, head=0.16,
                 band_gap=0.07, lam_w=0.42, label_pt=BASE_PT,
                 lam_col="#666666", arrow_col="#A32E1C", seed=0):
    """Bands read bottom to top, with a lambda column and arrows between levels."""
    mpl.rcParams.update(PAPER_RC)
    n = len(bands)
    thumb = (w_in - lam_w - gap * (n_per_row - 1)) / n_per_row
    h_in = n * (thumb + head) + band_gap * (n - 1)
    print(f"thumbnail {thumb*25.4:.0f}mm   figure {w_in:.2f} x {h_in:.2f}in")

    rng = np.random.default_rng(seed)
    fig = plt.figure(figsize=(w_in, h_in), facecolor="white")
    lam_y = []
    for j, (idx, label, lam) in enumerate(bands):        # j = 0 is the bottom band
        idx = np.asarray(idx)
        pick = rng.choice(idx, size=min(n_per_row, len(idx)), replace=False)
        y_img = j * (thumb + head + band_gap)
        fig.text(lam_w / w_in, (y_img + thumb + head * 0.28) / h_in, label,
                 ha="left", va="baseline", fontsize=label_pt)
        y_lam = (y_img + thumb * 0.5) / h_in
        lam_y.append(y_lam)
        fig.text((lam_w - 0.06) / w_in, y_lam, f"$\\lambda$ = {lam:.2f}",
                 ha="right", va="center", fontsize=label_pt - 0.5, color=lam_col)
        for col, i in enumerate(pick):
            ax = fig.add_axes([(lam_w + col * (thumb + gap)) / w_in, y_img / h_in,
                               thumb / w_in, thumb / h_in])
            ax.imshow(center_crop_square(Image.open(paths[i])).convert("L"), cmap="gray")
            ax.set_xticks([]); ax.set_yticks([])
            for s in ax.spines.values():
                s.set_linewidth(0.3); s.set_color("#cccccc")

    x_a = (lam_w - 0.17) / w_in
    pad = (label_pt * 0.9 / 72) / h_in
    for j in range(n - 1):
        y0, y1 = lam_y[j] + pad, lam_y[j + 1] - pad
        if y1 > y0:
            fig.patches.append(FancyArrowPatch(
                (x_a, y0), (x_a, y1), transform=fig.transFigure, figure=fig,
                arrowstyle="-|>", mutation_scale=6, color=arrow_col,
                lw=0.8, zorder=1000))

    save_fig(out, pad=0.01)
    mpl.rcParams.update(mpl.rcParamsDefault)


print("band_figure, climb_figure defined")

## Figure 1 — a photograph from the collection (Data section)

Reproduced whole rather than centre-cropped. `DATA_IDX` was found with
`show_shed_indexed(9115)`; it is the cowboys photograph, whose Europeana record
names the boy in the foreground as the photographer's son.

In [ ]:
DATA_IDX = 1499

mpl.rcParams.update(PAPER_RC)
img = Image.open(paths[DATA_IDX]).convert("L")
h_in = FIG_W * img.height / img.width
print(f"[{DATA_IDX}] {paths[DATA_IDX]}")

fig = plt.figure(figsize=(FIG_W, h_in), facecolor="white")
ax = fig.add_axes([0, 0, 1, 1])
ax.imshow(img, cmap="gray")
ax.set_xticks([]); ax.set_yticks([])
for s in ax.spines.values():
    s.set_linewidth(0.4); s.set_color("#bbbbbb")

save_fig(f"fig_data_example_{SUFFIX}", pad=0.01)
mpl.rcParams.update(mpl.rcParamsDefault)

## Figure 2 — photographs per year (Data section)

Dates come from the Europeana records. The collection spans fifty years but is
heavily concentrated in the 1950s, consistent with what is known about
Massafont's biography and with the broader history of commercial street
photography.

In [ ]:
YEAR_COL = "date"

meta = pd.read_csv(METADATA_CSV)
years = pd.to_datetime(meta[YEAR_COL], errors="coerce").dt.year.dropna().astype(int)
counts = years.value_counts().sort_index()
print(f"{len(years)} dated records, {years.min()}-{years.max()}; "
      f"peak {counts.idxmax()} ({counts.max()} photographs)")

mpl.rcParams.update(PAPER_RC)
fig, ax = plt.subplots(figsize=(FIG_W, 1.9), facecolor="white")
ax.bar(counts.index, counts.values, width=0.8, color="#9FE1CB",
       edgecolor="#0F6E56", lw=0.4)
ax.set_xlabel("year", labelpad=2)
ax.set_ylabel("photographs")
ax.set_xlim(counts.index.min() - 1, counts.index.max() + 1)
ax.tick_params(axis="x", labelsize=BASE_PT - 1)
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)

plt.tight_layout(pad=0.2)
save_fig(f"fig_photos_per_year_{SUFFIX}")
mpl.rcParams.update(mpl.rcParamsDefault)

## Figure 3 — the dendrogram

NB: figure 2 is in the Zenodo repository

The whole hierarchy in one figure. Filled circles are the 72 selected clusters,
sized by membership; small open circles below them are the groupings HDBSCAN
found inside a cluster but did not report; diamonds above them are merge points;
red ticks mark where photographs left the tree, their *length* proportional to
the number lost.

Goes into the paper as a rotated full-page `sidewaysfigure*`.

In [ ]:
SUB_YCLIP = None        # set to e.g. 99 to trim the high-lambda tail
SUB_COL   = "#B9B9B9"
MERGE_S, SUB_S = 9, 14
SHED_COL = "#A32E1C"
SHED_LW  = 1.1
SHED_HALFW_MIN, SHED_HALFW_MAX = 0.12, 1.30

mpl.rcParams.update(PAPER_RC)
fig, ax = plt.subplots(figsize=(FIG_W_WIDE, 5.6), facecolor="white")

# structure above the cut
for p, kids in children_of.items():
    py = node_height[p]
    xs = [x_pos[k] for k in kids]
    ax.plot([min(xs), max(xs)], [py, py], color="#888888", lw=0.45)
    for k in kids:
        ax.plot([x_pos[k]] * 2, [node_height[k], py], color="#888888", lw=0.45)

# structure below the cut
for p, kids in sub_children_of.items():
    py, ky = birth_of(p), min(sub_height[k] for k in kids)
    xs = [x_pos[k] for k in kids]
    ax.plot([x_pos[p]] * 2, [py, ky], color=SUB_COL, lw=0.35, zorder=1)
    ax.plot([min(xs), max(xs)], [ky, ky], color=SUB_COL, lw=0.35, zorder=1)
    for k in kids:
        ax.plot([x_pos[k]] * 2, [ky, sub_height[k]], color=SUB_COL, lw=0.35, zorder=1)

# shed marks: tick length carries the number lost
shed_counts = {n: c for n, c in noise_df.groupby("node").size().to_dict().items()
               if n in node_height}
smax = max(shed_counts.values())
for nid, c in sorted(shed_counts.items(), key=lambda kv: kv[1]):
    hw = SHED_HALFW_MIN + (SHED_HALFW_MAX - SHED_HALFW_MIN) * (c / smax)
    ax.plot([x_pos[nid] - hw, x_pos[nid] + hw], [node_height[nid]] * 2,
            color=SHED_COL, lw=SHED_LW, solid_capstyle="butt", zorder=2)

for p in children_of:                                   # merge points
    ax.scatter([x_pos[p]], [node_height[p]], marker="D", s=1.6,
               facecolor="#DDDDDD", edgecolor="#999999", lw=0.25, zorder=3)

for nid in sub_size:                                    # unselected sub-clusters
    ax.scatter([x_pos[nid]], [sub_height[nid]], s=2.6, facecolor="white",
               edgecolor="#7FA79B", lw=0.35, zorder=4)

for L in leaf_order:                                    # the jagged cut
    nid = cluster_node_id[L]
    ax.scatter([x_pos[nid]], [birth_height[L]],
               s=3.0 + 0.10 * np.sum(labels == L),
               facecolor="#9FE1CB", edgecolor="#0F6E56", lw=0.4, zorder=5)

ax.legend(handles=[
    Line2D([], [], marker="o", ls="", markersize=4.5, markerfacecolor="#9FE1CB",
           markeredgecolor="#0F6E56", markeredgewidth=0.5,
           label="selected cluster (area $\\propto$ images)"),
    Line2D([], [], marker="o", ls="", markersize=2.2, markerfacecolor="white",
           markeredgecolor="#7FA79B", markeredgewidth=0.5,
           label="unselected sub-cluster"),
    Line2D([], [], marker="D", ls="", markersize=1.8, markerfacecolor="#DDDDDD",
           markeredgecolor="#999999", markeredgewidth=0.4, label="merge point"),
    Line2D([], [], color=SHED_COL, lw=SHED_LW,
           label=f"noise shed (length $\\propto$ number, max {smax})"),
], loc="lower right", frameon=False, handlelength=2.2,
   borderpad=0.2, labelspacing=0.5)

ax.set_xticks([x_pos[cluster_node_id[L]] for L in leaf_order])
ax.set_xticklabels(leaf_order, rotation=90)
ax.invert_yaxis()
if SUB_YCLIP is not None:
    ax.set_ylim(np.percentile(list(sub_height.values()), SUB_YCLIP), 0.0)
ax.set_ylabel("$\\lambda$")
ax.set_xlabel("cluster", labelpad=2)
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)

plt.tight_layout(pad=0.2)
save_fig(f"fig01_dendrogram_{SUFFIX}")
mpl.rcParams.update(mpl.rcParamsDefault)

## Figure 4 — the most specific division

Clusters 6 and 7, both born at lambda 7.68: portraits of matadors, and people
photographed with them.

In [ ]:
band_figure(
    [(np.where(labels == L)[0],
      f"Cluster {L}: {name} \u2014 {int(np.sum(labels == L))} photographs, "
      f"\u03bb = {birth_height[L]:.2f}")
     for L, name in [(6, "portraits of matadors"),
                     (7, "people photographed with matadors")]],
    out=f"fig02_matadors_{SUFFIX}", seed=7)

## Figure 5 — the least specific clusters

Cluster 0 (pool), 1 (hospital) and 2 (sports), each separating from the rest of
the collection near the root. Size and lambda are pulled from the data so the
headings cannot drift out of step with the numbers in the text.

In [ ]:
band_figure(
    [(np.where(labels == L)[0],
      f"Cluster {L}: {name} \u2014 {int(np.sum(labels == L))} photographs, "
      f"\u03bb = {birth_height[L]:.2f}")
     for L, name in [(0, "swimming pool"), (1, "hospital"), (2, "sports")]],
    out=f"fig03_nearroot_{SUFFIX}", seed=11)

## Figure 6 — the gender division inside cluster 1

The two groupings HDBSCAN found inside the hospital cluster but did not report.
Run the cell, look at the two bands, then replace the generic labels with
"male patients" and "female patients" in the order they came out.

In [ ]:
kids = sorted(sub_children_of[cluster_node_id[1]], key=lambda k: sub_height[k])
for k in kids:
    print(f"node {k}: {sub_size[k]} images, \u03bb = {sub_height[k]:.2f}")

# after looking at the two bands, replace the descriptions below
SUB_NAMES = {kids[0]: "sub-cluster", kids[1]: "sub-cluster"}

band_figure(
    [(sorted(node_point_set(k)),
      f"Cluster 1, {SUB_NAMES[k]} \u2014 {sub_size[k]} photographs, "
      f"\u03bb = {sub_height[k]:.2f}")
     for k in kids],
    out=f"fig_hospital_subclusters_{SUFFIX}", seed=5)

## Figure 7 — the load-bearing splits

One panel per split, colouring the two sides against the rest of the tree in
grey. The splits are nested, so later panels subdivide branches coloured in
earlier ones. Sub-structure and shed ticks are tinted by side as well.

In [ ]:
N_PANELS = 3
PAIRS = [("#D97706", "#2563EB"),   # amber  / blue
         ("#15803D", "#DC2626"),   # green  / red
         ("#7C3AED", "#EA580C")]   # violet / orange
BASE_COLOR, SUB_BASE = "#d4d4d4", "#e2e2e2"

mpl.rcParams.update(PAPER_RC)
shed_counts = {n: c for n, c in noise_df.groupby("node").size().to_dict().items()
               if n in node_height}
SMAX = max(shed_counts.values())


def plot_split(split_node, colors, ax):
    sides = [subtree_nodes(k) for k in children_of[split_node]]
    side_of = {nid: j for j, s in enumerate(sides) for nid in s}
    col = lambda nid: colors[side_of[nid]] if nid in side_of else BASE_COLOR

    for p, kids in children_of.items():
        py = node_height[p]
        xs = [x_pos[k] for k in kids]
        ax.plot([min(xs), max(xs)], [py, py], color=col(p), lw=0.5, zorder=2)
        for k in kids:
            ax.plot([x_pos[k]] * 2, [node_height[k], py], color=col(k), lw=0.5, zorder=2)

    for p, kids in sub_children_of.items():
        owner = cluster_node_id[sub_owner[kids[0]]]
        c = colors[side_of[owner]] if owner in side_of else SUB_BASE
        py, ky = birth_of(p), min(sub_height[k] for k in kids)
        xs = [x_pos[k] for k in kids]
        ax.plot([x_pos[p]] * 2, [py, ky], color=c, lw=0.3, alpha=0.55, zorder=1)
        ax.plot([min(xs), max(xs)], [ky, ky], color=c, lw=0.3, alpha=0.55, zorder=1)
        for k in kids:
            ax.plot([x_pos[k]] * 2, [ky, sub_height[k]], color=c,
                    lw=0.3, alpha=0.55, zorder=1)
    for nid in sub_size:
        owner = cluster_node_id[sub_owner[nid]]
        c = colors[side_of[owner]] if owner in side_of else SUB_BASE
        ax.scatter([x_pos[nid]], [sub_height[nid]], s=1.6, facecolor="white",
                   edgecolor=c, lw=0.3, alpha=0.8, zorder=4)

    for nid, c in sorted(shed_counts.items(), key=lambda kv: kv[1]):
        hw = 0.12 + 1.18 * (c / SMAX)
        ax.plot([x_pos[nid] - hw, x_pos[nid] + hw], [node_height[nid]] * 2,
                color=col(nid), lw=0.9, solid_capstyle="butt", alpha=0.75, zorder=3)

    for L in leaf_order:
        nid = cluster_node_id[L]
        ax.scatter([x_pos[nid]], [birth_height[L]],
                   s=2.0 + 0.09 * np.sum(labels == L), facecolor=col(nid),
                   edgecolor="white", lw=0.25, alpha=0.95, zorder=5)

    sx_, sy_ = x_pos[split_node], node_height[split_node]
    ax.scatter([sx_], [sy_], s=45, facecolor="none", edgecolor="#333",
               lw=0.7, ls="--", zorder=10)

    ax.set_xticks([x_pos[cluster_node_id[L]] for L in leaf_order])
    ax.set_xticklabels(leaf_order, rotation=90)
    for t, L in zip(ax.get_xticklabels(), leaf_order):
        t.set_color(col(cluster_node_id[L]))
    ax.invert_yaxis()
    ax.set_ylabel("$\\lambda$")

    sizes = [n_img(k) for k in children_of[split_node]]
    ax.legend(handles=[
        Line2D([0], [0], color=colors[0], lw=2, label=f"{sizes[0]:,} photographs"),
        Line2D([0], [0], color=colors[1], lw=2, label=f"{sizes[1]:,} photographs"),
        Line2D([0], [0], color=BASE_COLOR, lw=2, label="outside this branch")],
        loc="lower left", frameon=False, handlelength=1.8,
        borderpad=0.15, labelspacing=0.35)
    ax.text(0.5, 1.02, f"Node {split_node}, $\\lambda$ = {sy_:.2f} \u2014 "
            f"{sum(sizes):,} photographs split {sizes[0]:,} / {sizes[1]:,}",
            transform=ax.transAxes, ha="center", va="bottom", fontsize=BASE_PT)
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)


nodes = list(splits.head(N_PANELS).node)
fig, axes = plt.subplots(len(nodes), 1, figsize=(FIG_W_WIDE, 1.55 * len(nodes)),
                         facecolor="white", sharex=True)
axes = np.atleast_1d(axes)
for ax, nid, pair in zip(axes, nodes, PAIRS):
    plot_split(int(nid), pair, ax)
for ax in axes[:-1]:
    ax.tick_params(axis="x", labelbottom=False)
axes[-1].set_xlabel("cluster", labelpad=2)

plt.tight_layout(pad=0.25, h_pad=1.4)
save_fig(f"fig_splits_top{N_PANELS}_{SUFFIX}")
mpl.rcParams.update(mpl.rcParamsDefault)

## Figure 8 — the two sides of the gender split

Node 9113: women, families and domestic scenes on one side; men, soldiers and
processions on the other. Sampled from the whole branch, not from single
clusters.

In [ ]:
SPLIT_NODE = int(splits.iloc[1].node)
kids = sorted(children_of[SPLIT_NODE], key=lambda k: -n_img(k))
for k in kids:
    ls = sorted(int(leaf_of_node[l]) for l in backbone_leaves(k))
    print(f"node {k}: {n_img(k)} photographs, clusters {ls}")

# name the two sides after checking which is which
BRANCH_NAMES = {kids[0]: "Women, families and domestic scenes",
                kids[1]: "Men, soldiers and processions"}

band_figure(
    [(branch_images(k),
      f"{BRANCH_NAMES[k]} \u2014 {n_img(k)} photographs, "
      f"\u03bb = {node_height[SPLIT_NODE]:.2f}")
     for k in kids],
    out=f"fig_gender_split_{SUFFIX}", seed=13)

## Figure 9 — climbing the tree from cluster 56

Read bottom to top: individual portraits of women, then group portraits, then
outdoor family scenes. Shows that families sit on the female side of a division
that otherwise separates men from women.

In [ ]:
START_CLUSTER, N_LEVELS = 56, 4

path = [cluster_node_id[START_CLUSTER]]
while len(path) <= N_LEVELS and path[-1] in child_to_parent:
    path.append(child_to_parent[path[-1]])

BANDS_C = []
for i, nid in enumerate(path):
    ls, idx = backbone_leaves(nid), branch_images(nid)
    if i == 0:
        lbl, lam = (f"Cluster {START_CLUSTER} \u2014 {len(idx)} photographs",
                    birth_height[START_CLUSTER])
    else:
        lbl, lam = (f"Node {nid} \u2014 {len(ls)} clusters, {len(idx)} photographs",
                    node_height[nid])
    BANDS_C.append((idx, lbl, lam))
    print(f"{lam:5.2f}  {lbl}")

climb_figure(BANDS_C, out=f"fig_climb_cluster{START_CLUSTER}_{SUFFIX}", seed=17)

## Figures 10 and 11 — the two ends of the attachment range

Node 9031: the highest junction at which a substantial body of images leaves,
with 64 of the 72 clusters still below it. Node 9115: photographs that reached
the children's branch and joined none of its four clusters.

In [ ]:
def shed_images(node):
    return sorted(noise_df.loc[noise_df.node == node, "idx"].tolist())


for node, name, out, seed in [
        (9031, "Shed near the root", f"fig_noise_root_{SUFFIX}", 23),
        (9115, "Shed above the children's clusters", f"fig_noise_children_{SUFFIX}", 29)]:
    idx = shed_images(node)
    band_figure(
        [(idx, f"{name} (node {node}) \u2014 {len(idx)} photographs, "
               f"\u03bb = {node_height[node]:.2f}")],
        out=out, seed=seed)

# Interactive explorer

The dendrogram as a self-contained HTML file: every element can be clicked — a
selected cluster, a merge point above the cut, an unreported grouping below it,
or a shed tick — and shows a sample of up to 20 thumbnails with its size and
density. This is what interpretation actually ran on; the static figures above
are what it produced.

In [ ]:
N_THUMBS, THUMB_SIZE, GALLERY_SIZE = 20, 160, 150


def thumb_b64(p, size=THUMB_SIZE):
    try:
        img = center_crop_square(Image.open(p)).convert("L").resize((size, size))
        buf = BytesIO(); img.save(buf, format="JPEG", quality=70)
        return base64.b64encode(buf.getvalue()).decode("ascii")
    except Exception:
        return None


def node_members(nid):
    if nid in sub_size:
        return sorted(node_point_set(nid))
    if nid in leaf_of_node:
        return list(np.where(labels == leaf_of_node[nid])[0])
    return branch_images(nid)


# ---- thumbnails for clusters, merge points and sub-clusters ------------
node_thumbs, node_meta = {}, {}
for nid in set(cluster_node_id.values()) | set(children_of) | set(sub_size):
    idx = node_members(nid)
    if not idx:
        continue
    sample = random.sample(list(idx), min(N_THUMBS, len(idx)))
    tl = [t for t in (thumb_b64(paths[i]) for i in sample) if t]
    if not tl:
        continue
    L = leaf_of_node.get(nid)
    kind = "sub" if nid in sub_size else ("leaf" if nid in leaf_of_node else "internal")
    node_thumbs[str(nid)] = tl
    node_meta[str(nid)] = {
        "kind": kind,
        "cluster": int(L) if L is not None else (int(sub_owner[nid]) if kind == "sub" else None),
        "depth": int(sub_depth_of[nid]) if kind == "sub" else None,
        "size": len(idx), "height": round(birth_of(nid), 2),
        "shed": int((noise_df.node == nid).sum())}

# ---- thumbnails for the photographs shed at each junction --------------
shed_counts = {int(n): int(c) for n, c in
               noise_df.groupby("node").size().to_dict().items() if n in node_height}
for nid, c in shed_counts.items():
    idx = noise_df.loc[noise_df.node == nid, "idx"].tolist()
    sample = random.sample(idx, min(N_THUMBS, len(idx)))
    tl = [t for t in (thumb_b64(paths[i]) for i in sample) if t]
    if not tl:
        continue
    ls = sorted(int(leaf_of_node[l]) for l in backbone_leaves(nid))
    node_thumbs[f"shed{nid}"] = tl
    node_meta[f"shed{nid}"] = {
        "kind": "shed", "size": c, "height": round(node_height[nid], 2),
        "beneath": len(ls),
        "clusters": ",".join(map(str, ls[:14])) + ("\u2026" if len(ls) > 14 else ""),
        "clustered": int(sum(int(np.sum(labels == L)) for L in ls))}

print(f"encoded thumbnails for {len(node_thumbs)} nodes "
      f"({sum(1 for m in node_meta.values() if m['kind'] == 'sub')} below the cut, "
      f"{len(shed_counts)} shedding junctions)")

In [ ]:
SVG_W, SVG_H = 1600, 820
ML, MR, MT, MB = 60, 40, 30, 90
xs_all = list(x_pos.values())
hs_all = (list(node_height.values()) + [birth_height[L] for L in cluster_node_id]
          + list(sub_height.values()))
xmin, xmax, hmax = min(xs_all), max(xs_all), max(hs_all)
sx = lambda x: ML + (x - xmin) / (xmax - xmin + 1e-9) * (SVG_W - ML - MR)
sy = lambda h: MT + h / (hmax + 1e-9) * (SVG_H - MT - MB)
XSCALE = (SVG_W - ML - MR) / (xmax - xmin + 1e-9)
bottom = sy(hmax)

parts = []
for p, kids in children_of.items():
    py = sy(node_height[p]); kxs = [sx(x_pos[k]) for k in kids]
    parts.append(f'<line x1="{min(kxs):.1f}" y1="{py:.1f}" x2="{max(kxs):.1f}" '
                 f'y2="{py:.1f}" class="branch"/>')
    for k in kids:
        parts.append(f'<line x1="{sx(x_pos[k]):.1f}" y1="{sy(node_height[k]):.1f}" '
                     f'x2="{sx(x_pos[k]):.1f}" y2="{py:.1f}" class="branch"/>')

for p, kids in sub_children_of.items():
    py = sy(birth_of(p)); ky = sy(min(sub_height[k] for k in kids))
    kxs = [sx(x_pos[k]) for k in kids]
    parts.append(f'<line x1="{sx(x_pos[p]):.1f}" y1="{py:.1f}" '
                 f'x2="{sx(x_pos[p]):.1f}" y2="{ky:.1f}" class="subbranch"/>')
    parts.append(f'<line x1="{min(kxs):.1f}" y1="{ky:.1f}" x2="{max(kxs):.1f}" '
                 f'y2="{ky:.1f}" class="subbranch"/>')
    for k in kids:
        parts.append(f'<line x1="{sx(x_pos[k]):.1f}" y1="{ky:.1f}" '
                     f'x2="{sx(x_pos[k]):.1f}" y2="{sy(sub_height[k]):.1f}" '
                     f'class="subbranch"/>')

smax = max(shed_counts.values())
for nid, c in sorted(shed_counts.items(), key=lambda kv: kv[1]):
    key = f"shed{nid}"
    if key not in node_thumbs:
        continue
    hw = (0.10 + 1.20 * (c / smax)) * XSCALE
    x, y = sx(x_pos[nid]), sy(node_height[nid])
    parts.append(f'<line x1="{x-hw:.1f}" y1="{y:.1f}" x2="{x+hw:.1f}" y2="{y:.1f}" '
                 f'class="node shedhit" data-node="{key}"/>')
    parts.append(f'<line x1="{x-hw:.1f}" y1="{y:.1f}" x2="{x+hw:.1f}" y2="{y:.1f}" '
                 f'class="shed" data-for="{key}"/>')

for p in children_of:
    if str(p) in node_thumbs:
        x, y = sx(x_pos[p]), sy(node_height[p])
        parts.append(f'<rect x="{x-3:.1f}" y="{y-3:.1f}" width="6" height="6" '
                     f'transform="rotate(45 {x:.1f} {y:.1f})" class="node internal" '
                     f'data-node="{p}"/>')

for L, nid in cluster_node_id.items():
    x, y = sx(x_pos[nid]), sy(birth_height[L])
    r = min(1.5 + 0.35 * np.sqrt(int(np.sum(labels == L))), 6)
    if nid not in sub_children_of:
        parts.append(f'<line x1="{x:.1f}" y1="{y:.1f}" x2="{x:.1f}" '
                     f'y2="{bottom:.1f}" class="connector"/>')
    parts.append(f'<circle cx="{x:.1f}" cy="{y:.1f}" r="{r:.1f}" '
                 f'class="node leaf" data-node="{nid}"/>')

for nid, sz in sub_size.items():
    x, y = sx(x_pos[nid]), sy(sub_height[nid])
    r = min(1.2 + 0.35 * np.sqrt(int(sz)), 5)
    parts.append(f'<circle cx="{x:.1f}" cy="{y:.1f}" r="{r:.1f}" '
                 f'class="node sub" data-node="{nid}"/>')

svg_body = "\n".join(parts)
thumbs_json, meta_json = json.dumps(node_thumbs), json.dumps(node_meta)
n_clusters_actual, n_sub = len(cluster_node_id), len(sub_size)
n_with_sub = len(sub_children_of.keys() & set(cluster_node_id.values()))
n_shed_total, n_shed_nodes = int(sum(shed_counts.values())), len(shed_counts)
print(f"svg built: {len(parts)} elements")

In [ ]:
html = f"""<!DOCTYPE html>
<html><head><meta charset="utf-8"><title>Massafont \u2014 HDBSCAN tree explorer</title>
<style>
 body {{ margin:0; font-family:-apple-system,Helvetica,Arial,sans-serif; background:#fff; color:#222; }}
 #header {{ padding:20px 24px 12px; border-bottom:1px solid #eee; }}
 #header h1 {{ font-size:16px; margin:0 0 6px; font-weight:600; }}
 #header p {{ font-size:12.5px; color:#555; margin:0; max-width:860px; line-height:1.5; }}
 #main {{ display:flex; }}
 #treeWrap {{ position:relative; flex:1; min-width:0; height:62vh; overflow:hidden;
   cursor:grab; border-bottom:1px solid #eee; }}
 #treeWrap.dragging {{ cursor:grabbing; }}
 svg {{ display:block; width:100%; height:100%; }}
 .branch {{ stroke:#ccc; stroke-width:1; fill:none; }}
 .subbranch {{ stroke:#d8d8d8; stroke-width:0.8; fill:none; }}
 .connector {{ stroke:#0F6E56; stroke-width:0.5; stroke-dasharray:3,2; opacity:0.3; }}
 .shed {{ stroke:#A32E1C; stroke-width:2.5; pointer-events:none; }}
 .shed.active {{ stroke:#6B1A0E; stroke-width:4; }}
 .shedhit {{ stroke:transparent; stroke-width:12; cursor:pointer; }}
 .node.leaf {{ fill:#9FE1CB; stroke:#0F6E56; stroke-width:1; cursor:pointer; }}
 .node.leaf:hover,.node.leaf.active {{ fill:#0F6E56; }}
 .node.sub {{ fill:#fff; stroke:#7FA79B; stroke-width:0.9; cursor:pointer; }}
 .node.sub:hover,.node.sub.active {{ fill:#7FA79B; }}
 .node.internal {{ fill:#ddd; stroke:#999; stroke-width:0.8; cursor:pointer; }}
 .node.internal:hover,.node.internal.active {{ fill:#999; }}
 #sidebar {{ width:210px; flex-shrink:0; border-left:1px solid #eee; padding:16px;
   display:flex; flex-direction:column; gap:14px; height:62vh; box-sizing:border-box; }}
 #sidebar h2 {{ font-size:11px; text-transform:uppercase; letter-spacing:.03em;
   color:#999; margin:0 0 8px; }}
 .controlRow {{ display:flex; align-items:center; gap:8px; }}
 .controlRow button {{ width:28px; height:28px; border:1px solid #ccc; background:#fff;
   border-radius:5px; cursor:pointer; font-size:13px; }}
 #sidebar ul {{ margin:0; padding-left:16px; font-size:11.5px; color:#666; line-height:1.9; }}
 #legendDot {{ display:inline-block; width:9px; height:9px; border-radius:50%;
   background:#9FE1CB; border:1px solid #0F6E56; }}
 #legendOpen {{ display:inline-block; width:9px; height:9px; border-radius:50%;
   background:#fff; border:1px solid #7FA79B; }}
 #legendDiamond {{ display:inline-block; width:8px; height:8px; background:#ddd;
   border:1px solid #999; transform:rotate(45deg); }}
 #legendShed {{ display:inline-block; width:16px; height:3px; background:#A32E1C;
   vertical-align:middle; }}
 #gallery {{ padding:16px 24px; }}
 #galleryLabel {{ font-size:13px; font-weight:600; margin-bottom:10px; color:#333; }}
 #galleryLabel .shedtag {{ color:#A32E1C; }}
 #galleryImgs {{ display:flex; flex-wrap:wrap; gap:8px; }}
 #galleryImgs img {{ width:{GALLERY_SIZE}px; height:{GALLERY_SIZE}px;
   object-fit:cover; border-radius:4px; }}
 #galleryImgs.shed img {{ outline:2px solid #A32E1C; outline-offset:-2px; }}
 #placeholder {{ font-size:12px; color:#aaa; }}
</style></head><body>
<div id="header">
 <h1>HDBSCAN's hierarchical structure \u2014 Massafont archive</h1>
 <p>Filled circles are the {n_clusters_actual} clusters HDBSCAN selected, each placed at
 the density (&lambda;) at which it was born; diamonds above them are merge points. Open
 circles below the cut are groupings the algorithm found but did not report \u2014
 {n_sub} of them, beneath {n_with_sub} clusters. Red ticks mark junctions where
 photographs left the tree without ever reaching a cluster: {n_shed_total} in total at
 {n_shed_nodes} junctions, tick length proportional to the number lost. Click any node
 or tick to see sample images below.</p>
</div>
<div id="main">
 <div id="treeWrap">
  <svg id="tree" viewBox="0 0 {SVG_W} {SVG_H}" preserveAspectRatio="xMidYMid meet">
   <g id="viewport">{svg_body}</g>
  </svg>
 </div>
 <div id="sidebar">
  <div><h2>Navigate</h2>
   <div class="controlRow">
    <button id="zoomIn">+</button><button id="zoomOut">&minus;</button>
    <button id="reset">&#10227;</button>
    <span style="font-size:11px;color:#999;">zoom / reset</span>
   </div>
   <ul><li>Scroll to zoom</li><li>Drag to pan</li><li>Hover to preview</li>
    <li>Click to pin</li></ul>
  </div>
  <div><h2>Legend</h2>
   <ul style="list-style:none;padding-left:0;">
    <li><span id="legendDot"></span>&nbsp; Selected cluster</li>
    <li><span id="legendOpen"></span>&nbsp; Unselected sub-cluster</li>
    <li><span id="legendDiamond"></span>&nbsp; Merge point</li>
    <li><span id="legendShed"></span>&nbsp; Photographs shed</li>
   </ul>
  </div>
 </div>
</div>
<div id="gallery">
 <div id="galleryLabel"><span id="placeholder">Hover a node or tick to see sample images</span></div>
 <div id="galleryImgs"></div>
</div>
<script>
const thumbs={thumbs_json}, meta={meta_json};
const viewport=document.getElementById('viewport'), wrap=document.getElementById('treeWrap');
const galleryLabel=document.getElementById('galleryLabel'),
      galleryImgs=document.getElementById('galleryImgs');
let scale=1,tx=0,ty=0;
function applyTransform(){{ viewport.setAttribute('transform',
  `translate(${{tx}},${{ty}}) scale(${{scale}})`); }}
function clampScale(s){{ return Math.max(s,1); }}
let dragging=false,lastX,lastY;
wrap.addEventListener('mousedown',e=>{{dragging=true;lastX=e.clientX;lastY=e.clientY;
  wrap.classList.add('dragging');}});
window.addEventListener('mouseup',()=>{{dragging=false;wrap.classList.remove('dragging');}});
window.addEventListener('mousemove',e=>{{ if(dragging){{ tx+=e.clientX-lastX;
  ty+=e.clientY-lastY; lastX=e.clientX; lastY=e.clientY; applyTransform(); }} }});
wrap.addEventListener('wheel',e=>{{ e.preventDefault();
  scale=clampScale(scale*(e.deltaY<0?1.1:0.9)); applyTransform(); }});
document.getElementById('zoomIn').onclick=()=>{{scale=clampScale(scale*1.2);applyTransform();}};
document.getElementById('zoomOut').onclick=()=>{{scale=clampScale(scale*0.8);applyTransform();}};
document.getElementById('reset').onclick=()=>{{scale=1;tx=0;ty=0;applyTransform();}};
let activeNode=null, activeShed=null;
function clearActive(){{
 if(activeNode) activeNode.classList.remove('active');
 if(activeShed) activeShed.classList.remove('active');
 activeNode=null; activeShed=null;
}}
function showNode(id,el){{
 const m=meta[id]; if(!m) return;
 clearActive();
 if(m.kind==='shed'){{
   const vis=document.querySelector(`.shed[data-for="${{id}}"]`);
   if(vis){{ vis.classList.add('active'); activeShed=vis; }}
 }} else {{ el.classList.add('active'); activeNode=el; }}
 let txt;
 if(m.kind==='leaf'){{
   txt=`Cluster ${{m.cluster}} \u2014 ${{m.size}} images \u2014 \u03bb=${{m.height}}`;
   if(m.shed) txt+=` \u2014 ${{m.shed}} shed here`;
 }} else if(m.kind==='sub'){{
   txt=`Unselected sub-cluster within cluster ${{m.cluster}} (level ${{m.depth}} below `
      +`the cut) \u2014 ${{m.size}} images \u2014 \u03bb=${{m.height}}`;
 }} else if(m.kind==='shed'){{
   txt=`<span class="shedtag">Shed at this junction \u2014 ${{m.size}} photographs, `
      +`never clustered \u2014 \u03bb=${{m.height}}</span> \u2014 above ${{m.beneath}} `
      +`cluster(s) holding ${{m.clustered}} images: ${{m.clusters}}`;
 }} else {{
   txt=`Merge of clusters below this point \u2014 ${{m.size}} images \u2014 \u03bb=${{m.height}}`;
   if(m.shed) txt+=` \u2014 ${{m.shed}} shed here`;
 }}
 galleryLabel.innerHTML = txt;
 galleryImgs.className = (m.kind==='shed') ? 'shed' : '';
 galleryImgs.innerHTML=(thumbs[id]||[]).map(b=>`<img src="data:image/jpeg;base64,${{b}}">`).join('');
}}
document.querySelectorAll('.node').forEach(n=>{{
 const id=n.getAttribute('data-node');
 n.addEventListener('mouseenter',()=>showNode(id,n));
 n.addEventListener('click',()=>showNode(id,n));
}});
applyTransform();
</script></body></html>
"""

out = f"hdbscan_tree_explorer_{SUFFIX}.html"
Path(out).write_text(html, encoding="utf-8")
print(f"saved -> {out}  ({len(html)/1e6:.1f} MB)")